# Stage 01 — Review (T1 + T3)

For every patent in `raw_images/`:
1. OCR each image → figure label (from the image itself)
2. Read BRIEF DESCRIPTION from `text/<patent_id>.txt` (written by Stage 00 from EPO)
3. Match OCR label → description line → assign `match_status`
4. Auto-fill visual taxonomy fields from description keywords
5. Write `labels/<patent_id>.json`
6. Render review table: thumbnail | OCR label | description | visual fields | status

**match_status colour key:**
- 🟢 `matched` — OCR label found and matched to a description line (confidence 0.95)
- 🔴 `unmatched` — OCR label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — OCR found nothing; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")


In [ ]:
from src.matcher import parse_description, match_images, label_from_filename
from src.cross_modal import load_siglip_model, verify_matches

# ── PatentSBERTa — semantic fallback for unmatched OCR labels ─────────────────
# Skipped gracefully if sentence-transformers is not installed.
sbert = None
try:
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa")
    print("PatentSBERTa ready.")
except ImportError:
    print("sentence-transformers not installed — semantic fallback disabled.")

# ── SigLIP — visual verification of text↔image matches ───────────────────────
# Set USE_SIGLIP = False for fast runs (skips ~3 GB model download).
USE_SIGLIP = False   # change to True once open_clip is installed
siglip_bundle = None
if USE_SIGLIP:
    try:
        siglip_bundle = load_siglip_model()
        print(f"SigLIP ready on {siglip_bundle[3]}.")
    except Exception as exc:
        print(f"SigLIP failed to load: {exc}")
        USE_SIGLIP = False
else:
    print("SigLIP disabled (USE_SIGLIP = False).")

In [ ]:
# ── Batch selector ────────────────────────────────────────────────────────
# Set BATCH_ID to the batch you want to review (see data/batches.xlsx Summary sheet).
BATCH_ID = 1

import pandas as pd
from pathlib import Path

cfg = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path = Path(cfg["paths"]["data"]) / "batches.xlsx"

if not batches_path.exists():
    raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

sheet_name  = f"Batch_{BATCH_ID:02d}"
batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")
print(batch_df[["patent_id","company_canonical","prototype_label"]].head(10).to_string(index=False))


In [ ]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
raw_dir    = cfg['paths']['raw_images']
text_dir   = cfg['paths']['text']
labels_dir = cfg['paths']['labels']
print(f"raw_images : {raw_dir}")
print(f"text       : {text_dir}")
print(f"labels     : {labels_dir}")

In [ ]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from text/<patent_id>.txt by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

In [ ]:
# ── Single-patent diagnostic (optional) ─────────────────────────────────────
# Set INSPECT_ID to any patent folder name to process just that one patent
# and see its JSON before running the full batch.
# Leave as None to skip.
INSPECT_ID = None  # e.g. "US2015014475A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    import json
    json_path = process_patent(
        INSPECT_ID, cfg, excel_index, raw_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(f"Wrote: {json_path}")
    print(f"Figures: {len(data.get('T3_images', []))}")
    print(f"has_splits: {data.get('has_splits')}")


In [ ]:
# ── Batch run configuration ──────────────────────────────────────────────────
LIMIT = None   # int or None — None processes all patents in raw_images/
# SigLIP on/off is controlled by USE_SIGLIP in the model-loading cell above.
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
from src.reviewer import run_stage01

summary_df = run_stage01(
    cfg,
    sbert_model   = sbert,           # loaded in model-loading cell above
    siglip_bundle = siglip_bundle,   # None if SigLIP not loaded
    skip_siglip   = not USE_SIGLIP,
    patent_ids    = patent_ids,   # process only this batch
)

In [ ]:
print(summary_df.to_string(index=False))

In [ ]:
# ── Generate per-patent review HTML files ─────────────────────────────────
# Creates review_html/Batch_{BATCH_ID:02d}/{patent_id}.html for every patent
# in this batch. Open each file in your browser — the wizard loads pre-filled
# with SigLIP T2 predictions and G1 architecture hint.
# Human confirms or corrects each field, then clicks "Export JSON & Next Patent".

from src.reviewer import build_patent_html
from pathlib import Path

HTML_TEMPLATE = repo_root / "UI_for_taxonomy_caracterization_10_0.html"
review_html_dir = Path(cfg["paths"]["data"]) / "review_html" / f"Batch_{BATCH_ID:02d}"
labels_dir      = Path(cfg["paths"]["labels"])

if not HTML_TEMPLATE.exists():
    print(f"⚠  HTML template not found at {HTML_TEMPLATE}")
    print("   Place UI_for_taxonomy_caracterization_10_0.html at repo root.")
else:
    generated = []
    for i, pid in enumerate(patent_ids, 1):
        label_json = labels_dir / f"{pid}.json"
        if not label_json.exists():
            print(f"  ⚠  No label JSON for {pid} — run Stage 01 first")
            continue
        html_path = build_patent_html(
            patent_id         = pid,
            label_json_path   = label_json,
            html_template_path= HTML_TEMPLATE,
            out_dir           = review_html_dir,
            pat_cur           = i,
            pat_tot           = len(patent_ids),
        )
        generated.append((pid, html_path))

    print(f"Generated {len(generated)} review HTML files → {review_html_dir}")
    print()
    # Print clickable file:// links for quick opening
    for pid, p in generated[:20]:
        print(f"  file://{p.resolve()}")
    if len(generated) > 20:
        print(f"  ... and {len(generated)-20} more")


In [ ]:
# ── Single-patent diagnostic — change INSPECT_ID to inspect any patent ─────────
import json
from pathlib import Path

INSPECT_ID = None   # e.g. "US2015014475A1"

if INSPECT_ID:
    p = Path(cfg["paths"]["labels"]) / f"{INSPECT_ID}.json"
    if p.exists():
        d    = json.loads(p.read_text())
        figs = d.get("T3_images", [])
        icons = {
            "matched":        "🟢",
            "semantic":       "🔵",
            "positional":     "🟡",
            "unmatched":      "🔴",
            "human_required": "⚫",
        }
        print(f"\n{INSPECT_ID}  |  splits={d.get('has_splits')}")
        print(f"{'FILE':<32} {'STATUS':<16} {'CONF':>5}  {'T2_PER':<20}  DESC[:50]")
        print("-" * 110)
        for f in figs:
            icon = icons.get(f.get("match_status", ""), "❓")
            t2   = f.get("T2_predictions") or {}
            per  = (t2.get("per") or {}).get("value", "—")
            desc = (f.get("matched_description") or "")[:50]
            conf = f.get("composite_confidence") or f.get("match_confidence") or 0.0
            print(
                f"{f['file']:<32} {icon} {f.get('match_status',''):<14} "
                f"{float(conf):>5.2f}  {per:<20}  {desc}"
            )
    else:
        print(f"No JSON found at {p}")

In [ ]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Run this after you have finished human review for this batch.
# Only patents with human_review_status == "approved" in batches.xlsx are copied.
# Raw files are never touched. Source is matched/{patent_id}/.

import shutil, json
from pathlib import Path
import pandas as pd

cfg           = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
matched_root  = Path(cfg["paths"]["matched"])
reviewed_root = Path(cfg["paths"]["reviewed"])
labels_dir    = Path(cfg["paths"]["labels"])
sheet_name    = f"Batch_{BATCH_ID:02d}"

batch_df = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
approved = batch_df[batch_df["human_review_status"].str.strip().str.lower() == "approved"]

copied  = 0
skipped = 0
errors  = 0

for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()

    src_dir  = matched_root / pid
    dest_dir = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠  matched/{pid}/ not found — skipping")
        skipped += 1
        continue

    dest_dir.mkdir(parents=True, exist_ok=True)

    for f in src_dir.glob("*.png"):
        dest_f = dest_dir / f.name
        if not dest_f.exists():
            shutil.copy2(f, dest_f)
            copied += 1
        else:
            skipped += 1

    # Also copy the label JSON if it exists
    label_json = labels_dir / f"{pid}.json"
    if label_json.exists():
        shutil.copy2(label_json, dest_dir / f"{pid}_labels.json")

print(f"Approved patents : {len(approved)}")
print(f"Files copied     : {copied}")
print(f"Already existed  : {skipped}")
print(f"Errors           : {errors}")
print(f"Output root      : {reviewed_root}")
